In [19]:
EMA_path = "./../data/datasets/manually_cleaned/approved_manually_cleaned/EMA.csv"
JAPAN_path = "./../data/datasets/manually_cleaned/approved_manually_cleaned/PMDA.csv"
SWISSMEDIC_path = "./../data/datasets/manually_cleaned/approved_manually_cleaned/Swissmedic.csv"
AUSTRALIA_path = "./../data/datasets/manually_cleaned/approved_manually_cleaned/TGA.csv"
FDA_path = "./../data/datasets/manually_cleaned/approved_manually_cleaned/FDA.csv"
HEALTHCANADA_path = "./../data/datasets/manually_cleaned/approved_manually_cleaned/HealthCanada.csv"


In [20]:
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
from matplotlib import cm
import matplotlib.colors as mcolors
import seaborn as sns
import json
import numpy as np
from matplotlib.ticker import FixedLocator, ScalarFormatter
import math
import os

In [21]:
# =============================
# CSV Loader
# =============================
def load_agency_csv(path: str, agency: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    # Spaltennamen bereinigen (falls irgendwo Leerzeichen sind)
    df.columns = df.columns.astype(str).str.strip()
    return df

# =============================
# Load all agencies into dfs
# =============================
df_ema = load_agency_csv(EMA_path, "EMA")
# df_fda = load_agency_csv(FDA_path, "FDA")
df_swissmedic = load_agency_csv(SWISSMEDIC_path, "SWISSMEDIC")
df_japan = load_agency_csv(JAPAN_path, "JAPAN")
df_australia = load_agency_csv(AUSTRALIA_path, "AUSTRALIA")
# df_healthcanada = load_agency_csv(HEALTHCANADA_path, "HEALTHCANADA")

# =============================
# Helper für Identifier-Zählung
# =============================
PLACEHOLDERS = {"not reported", "na", "n/a", "tbd", "none", ""}

def cleaned_series(s: pd.Series) -> pd.Series:
    x = s.astype("string").str.strip()
    x = x.mask(x.str.lower().isin(PLACEHOLDERS))
    return x

def agency_identifier_count(df: pd.DataFrame, agency: str) -> int:
    """
    Zählt den passenden Identifier pro Agency:
    - Japan: Anzahl Zeilen (= PDFs/Records in deinem CSV)
    - alle anderen: unique Marketing_authorisation_number (ohne Platzhalter)
    """
    if agency == "JAPAN":
        return len(df)

    s = cleaned_series(df["Marketing_authorisation_number"])
    return s.nunique(dropna=True)

# Results 1. Dataset Characteristics

Numbers per application

In [22]:
agencies = {
    # "FDA": df_fda,
    # "Health Canada": df_healthcanada,
    "EMA": df_ema,
    "Swissmedic": df_swissmedic,
    "Japan": df_japan,
    "Australia": df_australia,
}

# ============================================================
# 1) Record counts per agency (based on CSV rows)
# ============================================================
rows = []
total_records = sum(len(df) for df in agencies.values())

for name, df in agencies.items():
    n = len(df)
    pct = round(n / total_records * 100, 2) if total_records else 0.0
    rows.append({
        "Agency": name,
        "n_records": n,
        "%_of_overall_records": pct
    })

record_summary = pd.DataFrame(rows)

print("Record counts per agency (based on CSV rows)")
display(record_summary)

Record counts per agency (based on CSV rows)


,Agency,n_records,%_of_overall_records
0,EMA,1491,47.79
1,Swissmedic,233,7.47
2,Japan,408,13.08
3,Australia,988,31.67


#Number per Approval (Consolidated from Approved, Conditional Marketing Authorisation, Marketed)

In [23]:
# ============================================================
# Approved applications per agency (RECORD COUNTS, no normalisation)
# ============================================================

# Decisions, die als Approval zählen (exakt so geschrieben)
APPROVAL_DECISIONS = {
    "approved",
    "conditional marketing authorisation",
    "marketed",
}

rows = []
total_records_approved = 0

for name, df in agencies.items():
    # Anzahl Records insgesamt
    n_records = len(df)

    # Anzahl Records mit Approval-Decision
    n_approved = int(df["Decision"].isin(APPROVAL_DECISIONS).sum())

    rows.append({
        "Agency": name,
        "n_records_approved": n_approved,
    })

    total_records_approved += n_approved

approved_record_summary = pd.DataFrame(rows)

# Prozentanteil über alle Agencies (analog zu deinen anderen Tabellen)
approved_record_summary["%_of_overall_approved_records"] = (
    approved_record_summary["n_records_approved"] / total_records_approved * 100
).round(2)

print("Approved applications per agency (record counts, no normalisation)")
display(approved_record_summary)


Approved applications per agency (record counts, no normalisation)


,Agency,n_records_approved,%_of_overall_approved_records
0,EMA,1491,47.79
1,Swissmedic,233,7.47
2,Japan,408,13.08
3,Australia,988,31.67


Numbers per drug (unique Marketing_authorisation_number)

In [24]:
# ============================================================
# 2) Unique identifiers per agency (MA numbers; Japan = rows)
# ============================================================
def clean_ma_series(s: pd.Series) -> set:
    return set(
        s.astype("string")
         .str.strip()
         .str.lower()
         .mask(lambda x: x.isin(PLACEHOLDERS))
         .dropna()
         .unique()
    )

agency_ids = {}
overall_ids = 0

for name, df in agencies.items():
    if name == "Japan":
        n_ids = len(df)
        agency_ids[name] = n_ids
        overall_ids += n_ids
        continue

    ma_set = clean_ma_series(df["Marketing_authorisation_number"])
    n_ids = len(ma_set)
    agency_ids[name] = n_ids
    overall_ids += n_ids

rows = []
for name, n in agency_ids.items():
    pct = round(n / overall_ids * 100, 2) if overall_ids else 0.0
    rows.append({
        "Agency": name,
        "n_unique_identifiers": n,
        "%_of_overall": pct
    })

ma_summary = pd.DataFrame(rows)

print("Unique identifiers per agency (MA numbers; Japan = rows)")
display(ma_summary)

Unique identifiers per agency (MA numbers; Japan = rows)


,Agency,n_unique_identifiers,%_of_overall
0,EMA,1305,46.96
1,Swissmedic,195,7.02
2,Japan,408,14.68
3,Australia,871,31.34


# Study Flow Chart

In [25]:
import json
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# Paths
# --------------------------------------------------

BASE_MC = Path("./../data/datasets/manually_cleaned")
BASE_1995 = Path("./../data/datasets/1995")

ALL_DECISIONS = BASE_1995 / "all_decisions"
APPROVED = BASE_1995 / "approved"

# --------------------------------------------------
# Explicit agency mapping
# --------------------------------------------------
# key   = canonical agency name (used in tables / flowchart)
# value = filename stem in manually_cleaned

AGENCY_MAPPING = {
    "EMA": "EMA",
    "FDA": "FDA",
    "Health Canada": "HEALTHCANADA",
    "PMDA": "JAPAN",
    "TGA": "AUSTRALIA",
    "Swissmedic": "SWISSMEDIC",
}

PAR_AGENCIES = {"EMA", "PMDA", "TGA", "Swissmedic"}
NON_PAR_AGENCIES = {"FDA", "Health Canada"}

# --------------------------------------------------
# Helpers
# --------------------------------------------------

def count_manually_cleaned_json(path):
    """Count entries in manually cleaned JSON (dict-of-dicts)."""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return sum(1 for v in data.values() if isinstance(v, dict))

def count_csv(path):
    """Count rows in CSV."""
    return len(pd.read_csv(path))

def get_docname_and_decision_date_from_json(path):
    """
    Returns dict: {document_name: decision_date}
    """
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    out = {}
    for v in data.values():
        if isinstance(v, dict):
            doc = v.get("document_name")
            date = v.get("decision_date")
            if doc:
                out[doc] = date

    return out

def get_ids_from_csv(path):
    """
    Returns set of IDs contained in CSV
    Assumes there is a column called 'id'
    """
    df = pd.read_csv(path)
    return set(df["id"])

# --------------------------------------------------
# Per-agency sequential counts
# --------------------------------------------------

rows = []

for agency, mc_stem in AGENCY_MAPPING.items():
    mc_file = BASE_MC / f"{mc_stem}_manually_cleaned.json"
    all_file = ALL_DECISIONS / f"{agency.replace(' ', '')}.csv"
    approved_file = APPROVED / f"{agency.replace(' ', '')}.csv"

    n_retrieved = count_manually_cleaned_json(mc_file)
    n_all_1995 = count_csv(all_file)
    n_approved = count_csv(approved_file)

    rows.append({
        "Agency": agency,
        "Retrieved (manually cleaned)": n_retrieved,
        "Excluded (<1995)": n_retrieved - n_all_1995,
        "Evaluated (>=1995)": n_all_1995,
        "Excluded (not approved)": n_all_1995 - n_approved,
        "Approved": n_approved,
    })

per_agency_df = pd.DataFrame(rows)

# --------------------------------------------------
# Group-level summary (sequential)
# --------------------------------------------------

group_rows = []

for name, agencies in {
    "PAR agencies": PAR_AGENCIES,
    "FDA + Health Canada": NON_PAR_AGENCIES,
}.items():
    subset = per_agency_df[per_agency_df["Agency"].isin(agencies)]

    group_rows.append({
        "Group": name,
        "Retrieved": subset["Retrieved (manually cleaned)"].sum(),
        "Excluded (<1995)": subset["Excluded (<1995)"].sum(),
        "Evaluated (>=1995)": subset["Evaluated (>=1995)"].sum(),
        "Excluded (not approved)": subset["Excluded (not approved)"].sum(),
        "Approved": subset["Approved"].sum(),
    })

group_df = pd.DataFrame(group_rows)

# --------------------------------------------------
# OUTPUT
# --------------------------------------------------

print("\n=== Per-agency flowchart numbers ===")
display(per_agency_df)

print("\n=== Group-level flowchart numbers ===")
display(group_df)

print("\n=== Per-agency flowchart numbers ===")
display(per_agency_df)

print("\n=== Group-level flowchart numbers ===")
display(group_df)


# --------------------------------------------------
# DEBUG: manually_cleaned.csv vs all_decisions.csv
# --------------------------------------------------

print("\n=== DEBUG: PAR exclusions using manually_cleaned CSVs ===")

debug_rows = []

for agency in PAR_AGENCIES:
    mc_stem = AGENCY_MAPPING[agency]

    mc_csv = BASE_MC / f"{mc_stem}_manually_cleaned.csv"
    all_csv = ALL_DECISIONS / f"{agency.replace(' ', '')}.csv"

    df_mc = pd.read_csv(mc_csv)
    df_all = pd.read_csv(all_csv)

    # evaluable documents (>=1995 logic already applied in all_decisions)
    evaluable_docs = set(df_all["Document_name"])

    for _, row in df_mc.iterrows():
        doc = row.get("Document_name")

        if pd.notna(doc) and doc not in evaluable_docs:
            debug_rows.append({
                "Agency": agency,
                "Document_name": doc,
                "Decision_year": row.get("Decision_year"),
                "Decision_date": row.get("Decision_date"),
                "Decision": row.get("Decision"),
                "Current_status": row.get("Current_status"),
            })

debug_df = pd.DataFrame(debug_rows)
display(debug_df)

debug_df[["Decision_year", "Decision_date"]].apply(pd.unique)


=== Per-agency flowchart numbers ===


,Agency,Retrieved (manually cleaned),Excluded (<1995),Evaluated (>=1995),Excluded (not approved),Approved
0,EMA,1987,300,1687,196,1491
1,FDA,28288,9730,18558,0,18558
2,Health Canada,11523,1237,10286,0,10286
3,PMDA,408,0,408,0,408
4,TGA,1050,8,1042,54,988
5,Swissmedic,234,0,234,1,233



=== Group-level flowchart numbers ===


,Group,Retrieved,Excluded (<1995),Evaluated (>=1995),Excluded (not approved),Approved
0,PAR agencies,3679,308,3371,251,3120
1,FDA + Health Canada,39811,10967,28844,0,28844



=== Per-agency flowchart numbers ===


...


=== Group-level flowchart numbers ===


,Group,Retrieved,Excluded (<1995),Evaluated (>=1995),Excluded (not approved),Approved
0,PAR agencies,3679,308,3371,251,3120
1,FDA + Health Canada,39811,10967,28844,0,28844



=== DEBUG: PAR exclusions using manually_cleaned CSVs ===


,Agency,Document_name,Decision_year,Decision_date,Decision,Current_status
0,EMA,mabcampath-epar-scientific-discussion_en.pdf,Not reported,Not reported,Not reported,withdrawn
1,EMA,photobarr-epar-scientific-discussion_en.pdf,Not reported,Not reported,approved,withdrawn
2,EMA,contusugene-ladenovec-gendux-withdrawal-assess...,Not reported,Not reported,withdrawn,withdrawn
3,EMA,doxorubicin-sun-withdrawal-assessment-report_e...,Not reported,Not reported,refused,NaN
4,EMA,inductos-epar-scientific-discussion_en.pdf,Not reported,Not reported,approved,authorised
...,...,...,...,...,...,...
309,TGA,auspar-mvc-covid-19-vaccine-230907.pdf,Not applicable,Not applicable,withdrawn,withdrawn
310,TGA,auspar-nivolumab-ipilimumab-210427.pdf,Not applicable,Not applicable,withdrawn,withdrawn
311,TGA,auspar-alprostadil-160609.pdf,Not reported,Not reported,Withdrawn,Withdrawn
312,TGA,auspar-tecentriq-220824.pdf,Not applicable,Not applicable,withdrawn,withdrawn


,Decision_year,Decision_date
0,Not reported,Not reported
1,Error: The response is not valid JSON: ```json...,Error: The response is not valid JSON: ```json...
2,Error: The response is not valid JSON: ```json...,Error: The response is not valid JSON: ```json...
3,Error: The response is not valid JSON: ```json...,Error: The response is not valid JSON: ```json...
4,Error: The response is not valid JSON: ```json...,Error: The response is not valid JSON: ```json...
5,Error: The response is not valid JSON: ```json...,Error: The response is not valid JSON: ```json...
6,Error: The response is not valid JSON: ```json...,Error: The response is not valid JSON: ```json...
7,Not applicable,Not applicable


# Table 1: Key characteristics of approvals

In [26]:
# ============================================================
# Constants & helpers (analysis)
# ============================================================
DRUG_CLASS_ORDER = [
    "small molecule",
    "biologics",
    "cell and gene therapy",
    "peptides and proteins",
    "vaccine",
    "not reported",
    "other",
]

agencies = {
    "EMA": pd.read_csv(EMA_path),
    "PMDA": pd.read_csv(JAPAN_path),
    "Swissmedic": pd.read_csv(SWISSMEDIC_path),
    "TGA": pd.read_csv(AUSTRALIA_path),
    "FDA": pd.read_csv(FDA_path),
    "Health Canada": pd.read_csv(HEALTHCANADA_path),
}

NON_PAR_AGENCIES = {"FDA", "Health Canada"}

def norm_series(s: pd.Series) -> pd.Series:
    """Strip whitespace, lower-case, keep NaN."""
    return s.astype("string").str.strip().str.lower()

# ============================================================
# Decision distribution (record-level, per agency)
# Output wie in deinem Screenshot (inkl. consolidated)
# ============================================================
def decision_distribution(df: pd.DataFrame) -> pd.DataFrame:
    n = len(df)

    decision = (
        df.get("Decision", pd.Series([pd.NA] * n))
          .astype("string")
          .str.strip()
          .str.lower()
    )

    # Platzhalter -> NA (damit <NA> separat erscheint)
    decision = decision.mask(decision.isin(PLACEHOLDERS), pd.NA)

    counts = decision.value_counts(dropna=False)
    dist = counts.reset_index()
    dist.columns = ["Decision", "n"]
    dist["%"] = (dist["n"] / n * 100).round(2) if n else 0.0

    consolidated_mask = decision.isin({
        "approved",
        "conditional marketing authorisation",
        "conditional marketing authorization",
    })

    consolidated_row = pd.DataFrame([{
        "Decision": "consolidated (approved + conditional marketing authorisation)",
        "n": int(consolidated_mask.sum()),
        "%": round(consolidated_mask.sum() / n * 100, 2) if n else 0.0
    }])

    return pd.concat([dist, consolidated_row], ignore_index=True)

# ============================================================
# Drug class summary (record-level)
# ============================================================
def bucket_drug_class(x) -> str:
    if pd.isna(x):
        return "not reported"

    t = str(x).strip().lower()
    if t in PLACEHOLDERS:
        return "not reported"

    if t in DRUG_CLASS_ORDER:
        return t

    return "other"

def drug_class_summary(df: pd.DataFrame) -> pd.DataFrame:
    n = len(df)
    s = df.get("Drug_class", pd.Series([pd.NA] * n)).map(bucket_drug_class)
    counts = s.value_counts()

    rows = []
    for c in DRUG_CLASS_ORDER:
        k = int(counts.get(c, 0))
        rows.append({
            "Drug class": c,
            "n": k,
            "%": round(k / n * 100, 2) if n else 0.0
        })
    return pd.DataFrame(rows)

# ============================================================
# Therapeutic area composition (Top 5, mention-based)
# ============================================================
def therapeutic_area_top5(df: pd.DataFrame) -> pd.DataFrame:
    col = "Disease_class(es)"
    if col not in df.columns:
        return pd.DataFrame(columns=["Therapeutic area", "n", "%"])

    s = norm_series(df[col]).mask(lambda x: x.isin(PLACEHOLDERS)).dropna()
    if s.empty:
        return pd.DataFrame(columns=["Therapeutic area", "n", "%"])

    exploded = (
        s.str.split(";")
         .explode()
         .astype("string")
         .str.strip()
         .str.lower()
    )
    exploded = exploded[~exploded.isin(PLACEHOLDERS)].dropna()

    counts = exploded.value_counts().head(10)  # Top 10
    denom = int(exploded.shape[0])

    out = counts.reset_index()
    out.columns = ["Therapeutic area", "n"]
    out["%"] = (out["n"] / denom * 100).round(2) if denom else 0.0
    return out

FOCUS_DISEASE_CLASSES = [
    "diseases of the circulatory system",
    "diseases of the nervous system",
    "neoplasms",
    "endocrine, nutritional and metabolic diseases",
    "infectious diseases",
]

def therapeutic_area_focus5(df: pd.DataFrame) -> pd.DataFrame:
    col = "Disease_class(es)"
    if col not in df.columns:
        return pd.DataFrame(columns=["Therapeutic area", "n", "%"])

    s = (
        df[col]
        .astype("string")
        .str.strip()
        .str.lower()
        .mask(lambda x: x.isin(PLACEHOLDERS))
        .dropna()
    )
    if s.empty:
        return pd.DataFrame(
            [{"Therapeutic area": c, "n": 0, "%": 0.0} for c in FOCUS_DISEASE_CLASSES]
        )

    exploded = (
        s.str.split(";")
         .explode()
         .astype("string")
         .str.strip()
         .str.lower()
    )
    exploded = exploded[~exploded.isin(PLACEHOLDERS)].dropna()

    denom = int(exploded.shape[0])  # mention-based (wie Top 5)
    counts = exploded.value_counts()

    rows = []
    for c in FOCUS_DISEASE_CLASSES:
        n = int(counts.get(c, 0))
        pct = round(n / denom * 100, 2) if denom else 0.0
        rows.append({"Therapeutic area": c, "n": n, "%": pct})

    return pd.DataFrame(rows)


# ============================================================
# Run for all agencies
# ============================================================
for name, df in agencies.items():
    print("\n" + "=" * 70)
    print(f"{name} | Records: {len(df)}")

    print("\nDecision distribution (per application)")
    display(decision_distribution(df))

    print("\nDrug classes (applications)")
    display(drug_class_summary(df))

    print("\nTherapeutic area composition (Top 5)")
    display(therapeutic_area_top5(df))

# ============================================================
# OVERALL
# ============================================================
df_overall = pd.concat(
    [df for name, df in agencies.items() if name in PAR_AGENCIES],
    ignore_index=True
)

print("\n" + "=" * 70)
print("OVERALL (across PAR based agencies)")
print(f"Records: {len(df_overall)}")

print("\nDecision distribution (OVERALL)")
display(decision_distribution(df_overall))

print("\nDrug classes (OVERALL)")
display(drug_class_summary(df_overall))

print("\nTherapeutic area composition (Top 5, OVERALL)")
display(therapeutic_area_top5(df_overall))

print("\nTherapeutic area composition (selected 5 disease classes)")
display(therapeutic_area_focus5(df))

# ============================================================
# OVERALL (NON-PAR agencies only: FDA + Health Canada)
# ============================================================

df_nonpar_overall = pd.concat(
    [df for name, df in agencies.items() if name in NON_PAR_AGENCIES],
    ignore_index=True
)

print("\n" + "=" * 70)
print("OVERALL (Non-PAR agencies: FDA + Health Canada)")
print(f"Records: {len(df_nonpar_overall)}")

print("\nDecision distribution (OVERALL, Non-PAR)")
display(decision_distribution(df_nonpar_overall))

print("\nDrug classes (OVERALL, Non-PAR)")
display(drug_class_summary(df_nonpar_overall))

print("\nTherapeutic area composition (Top 5, OVERALL, Non-PAR)")
display(therapeutic_area_top5(df_nonpar_overall))

print("\nTherapeutic area composition (selected 5 disease classes, OVERALL, Non-PAR)")
display(therapeutic_area_focus5(df_nonpar_overall))



EMA | Records: 1491

Decision distribution (per application)


,Decision,n,%
0,approved,1459,97.85
1,conditional marketing authorisation,32,2.15
2,consolidated (approved + conditional marketing...,1491,100.0



Drug classes (applications)


,Drug class,n,%
0,small molecule,919,61.64
1,biologics,350,23.47
2,cell and gene therapy,27,1.81
3,peptides and proteins,89,5.97
4,vaccine,68,4.56
5,not reported,0,0.00
6,other,38,2.55



Therapeutic area composition (Top 5)


...

,Drug class,n,%
0,small molecule,25716,89.16
1,biologics,830,2.88
2,cell and gene therapy,7,0.02
3,peptides and proteins,1081,3.75
4,vaccine,100,0.35
5,not reported,346,1.20
6,other,764,2.65



Therapeutic area composition (Top 5, OVERALL, Non-PAR)


,Therapeutic area,n,%
0,diseases of the circulatory system,2420,12.17
1,diseases of the nervous system,2006,10.09
2,"endocrine, nutritional, and metabolic diseases",1727,8.68
3,diseases of the genitourinary system,1582,7.95
4,mental and behavioural disorders,1574,7.91
5,infectious and parasitic diseases,1549,7.79
6,neoplasms,1377,6.92
7,diseases of the digestive system,1353,6.8
8,diseases of the respiratory system,1344,6.76
9,diseases of the skin,1248,6.28



Therapeutic area composition (selected 5 disease classes, OVERALL, Non-PAR)


,Therapeutic area,n,%
0,diseases of the circulatory system,2420,12.17
1,diseases of the nervous system,2006,10.09
2,neoplasms,1377,6.92
3,"endocrine, nutritional and metabolic diseases",0,0.00
4,infectious diseases,0,0.00


# 4. Administration routes and pharmaceutical forms

In [27]:
import pandas as pd

# =========================
# Agency groups
# =========================

PAR_AGENCIES = {"EMA", "Swissmedic", "TGA", "PMDA"}
API_AGENCIES = {"FDA", "Health Canada"}

# =========================
# Load data
# =========================

df_fda = pd.read_csv(FDA_path)
df_health_canada = pd.read_csv(HEALTHCANADA_path)
df_ema = pd.read_csv(EMA_path)
df_swissmedic = pd.read_csv(SWISSMEDIC_path)
df_tga = pd.read_csv(AUSTRALIA_path)
df_pmda = pd.read_csv(JAPAN_path)

agencies = {
    "EMA": df_ema,
    "Swissmedic": df_swissmedic,
    "TGA": df_tga,
    "PMDA": df_pmda,
    "FDA": df_fda,
    "Health Canada": df_health_canada,
}

# =========================
# Placeholders
# =========================

PLACEHOLDERS = {"not reported", "na", "n/a", "none", ""}

# =========================
# Administration route (absolute)
# =========================

def administration_route_top10(df: pd.DataFrame) -> pd.DataFrame:
    col = "Administration_route"
    if col not in df.columns:
        return pd.DataFrame(columns=["Administration_route", "n", "%"])

    s = (
        df[col]
        .astype("string")
        .str.lower()
        .str.strip()
        .mask(lambda x: x.isin(PLACEHOLDERS))
        .dropna()
    )

    if s.empty:
        return pd.DataFrame(columns=["Administration_route", "n", "%"])

    exploded = (
        s.str.split(";")
        .explode()
        .astype("string")
        .str.strip()
    )
    exploded = exploded[~exploded.isin(PLACEHOLDERS)].dropna()

    exploded = exploded.replace({
        "intravenous": "parenteral",
        "subcutaneous": "parenteral",
        "intramuscular": "parenteral",
    })

    counts = exploded.value_counts().head(10)
    denom = len(exploded)

    out = counts.reset_index()
    out.columns = ["Administration_route", "n"]
    out["%"] = (out["n"] / denom * 100).round(2)
    return out

# =========================
# Pharmaceutical form (absolute)
# =========================

def pharmaceutical_form_top10(df: pd.DataFrame) -> pd.DataFrame:
    col = "Pharmaceutical_form"
    if col not in df.columns:
        return pd.DataFrame(columns=["Pharmaceutical_form", "n", "%"])

    s = (
        df[col]
        .astype("string")
        .str.lower()
        .str.strip()
        .mask(lambda x: x.isin(PLACEHOLDERS))
        .dropna()
    )

    if s.empty:
        return pd.DataFrame(columns=["Pharmaceutical_form", "n", "%"])

    exploded = (
        s.str.split(";")
        .explode()
        .astype("string")
        .str.strip()
    )
    exploded = exploded[~exploded.isin(PLACEHOLDERS)].dropna()

    exploded = exploded.replace({
        "tablet": "tablet / capsule",
        "capsule": "tablet / capsule",
        "solution": "solution / injectable",
        "injectable": "solution / injectable",
    })

    counts = exploded.value_counts().head(10)
    denom = len(exploded)

    out = counts.reset_index()
    out.columns = ["Pharmaceutical_form", "n"]
    out["%"] = (out["n"] / denom * 100).round(2)
    return out

# =========================
# Pharmaceutical form NORMALISED per Drug class
# =========================

def pharmaceutical_form_by_drug_class(df: pd.DataFrame) -> pd.DataFrame:
    required = {"Pharmaceutical_form", "Drug_class"}
    if not required.issubset(df.columns):
        return pd.DataFrame(
            columns=["Drug_class", "Pharmaceutical_form", "n", "%"]
        )

    df = df.copy()

    df["Pharmaceutical_form"] = (
        df["Pharmaceutical_form"]
        .astype("string")
        .str.lower()
        .str.strip()
        .mask(lambda x: x.isin(PLACEHOLDERS))
    )

    df["Drug_class"] = (
        df["Drug_class"]
        .astype("string")
        .str.lower()
        .str.strip()
        .mask(lambda x: x.isin(PLACEHOLDERS))
    )

    df = df.dropna(subset=["Pharmaceutical_form", "Drug_class"])

    df = df.assign(
        Pharmaceutical_form=df["Pharmaceutical_form"].str.split(";")
    ).explode("Pharmaceutical_form")

    df["Pharmaceutical_form"] = df["Pharmaceutical_form"].str.strip()

    df["Pharmaceutical_form"] = df["Pharmaceutical_form"].replace({
        "tablet": "tablet / capsule",
        "capsule": "tablet / capsule",
        "solution": "solution / injectable",
        "injectable": "solution / injectable",
        "suspension": "suspension",
    })

    counts = (
        df
        .groupby(["Drug_class", "Pharmaceutical_form"])
        .size()
        .reset_index(name="n")
    )

    counts["%"] = (
        counts["n"]
        / counts.groupby("Drug_class")["n"].transform("sum")
        * 100
    ).round(2)

    return counts.sort_values(
        ["Drug_class", "%"], ascending=[True, False]
    )

# =========================
# Per-agency output (UNCHANGED)
# =========================

for name, df in agencies.items():
    print("\n" + "=" * 70)
    print(name)

    print("\nAdministration route (Top 10)")
    display(administration_route_top10(df))

    print("\nPharmaceutical form (Top 10, absolute)")
    display(pharmaceutical_form_top10(df))

# =========================
# Build OVERALL datasets
# =========================

df_overall_par = pd.concat(
    [df for name, df in agencies.items() if name in PAR_AGENCIES],
    ignore_index=True
).drop_duplicates()

df_overall_api = pd.concat(
    [df for name, df in agencies.items() if name in API_AGENCIES],
    ignore_index=True
).drop_duplicates()

# =========================
# OVERALL outputs (UNCHANGED)
# =========================

print("\n" + "=" * 70)
print("Overall – PAR agencies")
display(administration_route_top10(df_overall_par))
display(pharmaceutical_form_by_drug_class(df_overall_par))

print("\n" + "=" * 70)
print("Overall – FDA & Health Canada")
display(administration_route_top10(df_overall_api))
display(pharmaceutical_form_by_drug_class(df_overall_api))

# =========================
# Paper summary:
# Top pharmaceutical form per Drug class
# =========================

def top_pharmaceutical_form_per_drug_class(
    df: pd.DataFrame,
    drug_class_patterns: dict
) -> pd.DataFrame:

    df_norm = pharmaceutical_form_by_drug_class(df)
    rows = []

    for label, pattern in drug_class_patterns.items():
        subset = df_norm[
            df_norm["Drug_class"].str.contains(pattern, na=False, regex=True)
        ]

        if subset.empty:
            continue

        top_row = subset.sort_values("n", ascending=False).iloc[0]

        rows.append({
            "Drug class": label,
            "Top pharmaceutical form": top_row["Pharmaceutical_form"],
            "n": int(top_row["n"]),
            "% (within drug class)": float(top_row["%"]),
        })

    return pd.DataFrame(rows)

DRUG_CLASS_PATTERNS = {
    "Small molecules": "small molecule",
    "Biologics & advanced therapies": "biologics|peptides|cell and gene",
    "Vaccines": "vaccin|immunological",
}

print("\n" + "=" * 70)
print("Paper summary – FDA & Health Canada")
display(
    top_pharmaceutical_form_per_drug_class(
        df_overall_api, DRUG_CLASS_PATTERNS
    )
)

print("\n" + "=" * 70)
print("Paper summary – PAR agencies")
display(
    top_pharmaceutical_form_per_drug_class(
        df_overall_par, DRUG_CLASS_PATTERNS
    )
)


EMA

Administration route (Top 10)


,Administration_route,n,%
0,oral,696,45.79
1,parenteral,674,44.34
2,inhalation,47,3.09
3,ocular,19,1.25
4,cutaneous,19,1.25
5,intravitreal,18,1.18
6,nasal,11,0.72
7,sublingual,4,0.26
8,intradermal,4,0.26
9,epilesional,4,0.26



Pharmaceutical form (Top 10, absolute)


,Pharmaceutical_form,n,%
0,tablet / capsule,681,43.05
1,solution / injectable,323,20.42
2,powder,220,13.91
3,concentrate,124,7.84
4,suspension,57,3.6
5,solvent,43,2.72
6,lyophilisate,24,1.52
7,dispersion,19,1.2
8,eye drops,13,0.82
9,spray,12,0.76



Swissmedic

Administration route (Top 10)


...

,Drug_class,Pharmaceutical_form,n,%
6,biologics,solution / injectable,383,43.57
1,biologics,injection,252,28.67
5,biologics,powder,155,17.63
2,biologics,kit,38,4.32
10,biologics,vial,29,3.30
...,...,...,...,...
117,vaccine,kit,3,2.88
123,vaccine,tablet / capsule,2,1.92
115,vaccine,dispersion,1,0.96
116,vaccine,emulsion,1,0.96



Paper summary – FDA & Health Canada


,Drug class,Top pharmaceutical form,n,% (within drug class)
0,Small molecules,tablet / capsule,16318,61.41
1,Biologics & advanced therapies,solution / injectable,383,43.57
2,Vaccines,suspension,44,42.31



Paper summary – PAR agencies


,Drug class,Top pharmaceutical form,n,% (within drug class)
0,Small molecules,tablet / capsule,1315,71.62
1,Biologics & advanced therapies,solution / injectable,418,45.19
2,Vaccines,suspension,123,54.91


# 6. Regulatory review durations

In [28]:
import pandas as pd
from IPython.display import display

# ============================================================
# Helper: compute review duration (POSITIVE ONLY)
# ============================================================
def compute_review_duration(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds 'review_duration_days' = Decision_date - Application_date.
    Keeps ONLY positive durations (> 0 days).
    """
    tmp = df.copy()

    tmp["Application_date"] = pd.to_datetime(
        tmp["Application_date"], errors="coerce", dayfirst=True
    )
    tmp["Decision_date"] = pd.to_datetime(
        tmp["Decision_date"], errors="coerce", dayfirst=True
    )

    tmp["review_duration_days"] = (
        tmp["Decision_date"] - tmp["Application_date"]
    ).dt.days

    # keep only positive durations
    tmp = tmp[tmp["review_duration_days"] > 0]

    return tmp.dropna(subset=["review_duration_days"])


# ============================================================
# Median, Q1, Q3, IQR (positive durations only)
# ============================================================
def review_duration_median_iqr(df: pd.DataFrame) -> pd.DataFrame:
    """
    Median, Q1, Q3 and IQR (Q3 - Q1) of positive review durations (days),
    stratified by abridged vs non-abridged.
    """
    tmp = compute_review_duration(df)

    tmp["Procedure"] = (
        tmp["Nonclinical_abridged"]
        .astype("string")
        .str.strip()
        .str.lower()
        .map({"yes": "Abridged", "no": "Non-abridged"})
    )

    tmp = tmp.dropna(subset=["Procedure"])

    out = (
        tmp.groupby("Procedure")["review_duration_days"]
        .agg(
            median_days="median",
            q1_days=lambda x: x.quantile(0.25),
            q3_days=lambda x: x.quantile(0.75),
        )
        .reset_index()
    )

    out["iqr_days"] = out["q3_days"] - out["q1_days"]

    return out


# ============================================================
# Min / Max (positive durations only)
# ============================================================
def review_duration_min_max(df: pd.DataFrame) -> pd.DataFrame:
    """
    Min and max positive review duration (days),
    stratified by abridged vs non-abridged.
    """
    tmp = compute_review_duration(df)

    tmp["Procedure"] = (
        tmp["Nonclinical_abridged"]
        .astype("string")
        .str.strip()
        .str.lower()
        .map({"yes": "Abridged", "no": "Non-abridged"})
    )

    tmp = tmp.dropna(subset=["Procedure"])

    out = (
        tmp.groupby("Procedure")["review_duration_days"]
        .agg(
            min_days="min",
            max_days="max",
        )
        .reset_index()
    )

    return out


# ============================================================
# Agencies to include (FDA & Health Canada excluded)
# ============================================================
agencies_timeline = {
    "EMA": df_ema,
    "Swissmedic": df_swissmedic,
    "PMDA": df_japan,
    "TGA": df_australia,
}

# ============================================================
# Per-agency outputs
# ============================================================
for name, df in agencies_timeline.items():
    print("\n" + "=" * 70)
    print(f"{name} – Review timelines (positive durations only)")

    print("\nMedian, Q1, Q3 and IQR (days)")
    display(review_duration_median_iqr(df))

    print("\nMin / Max (days)")
    display(review_duration_min_max(df))


# ============================================================
# OVERALL (across included agencies)
# ============================================================
df_overall_timeline = pd.concat(list(agencies_timeline.values()), ignore_index=True)

print("\n" + "=" * 70)
print("OVERALL – Review timelines (positive durations only)")

print("\nMedian, Q1, Q3 and IQR (days)")
display(review_duration_median_iqr(df_overall_timeline))

print("\nMin / Max (days)")
display(review_duration_min_max(df_overall_timeline))



EMA – Review timelines (positive durations only)

Median, Q1, Q3 and IQR (days)


,Procedure,median_days,q1_days,q3_days,iqr_days
0,Abridged,331.0,254.0,398.0,144.0
1,Non-abridged,378.0,329.0,451.0,122.0



Min / Max (days)


,Procedure,min_days,max_days
0,Abridged,9.0,1004.0
1,Non-abridged,2.0,1627.0



Swissmedic – Review timelines (positive durations only)

Median, Q1, Q3 and IQR (days)


...

,Procedure,min_days,max_days
0,Abridged,4.0,931.0
1,Non-abridged,32.0,1045.0



OVERALL – Review timelines (positive durations only)

Median, Q1, Q3 and IQR (days)


,Procedure,median_days,q1_days,q3_days,iqr_days
0,Abridged,324.0,242.0,398.5,156.5
1,Non-abridged,357.0,287.0,429.0,142.0



Min / Max (days)


,Procedure,min_days,max_days
0,Abridged,3.0,1004.0
1,Non-abridged,2.0,1627.0


# 7. Relationship between approval activity and disease incidence


In [29]:
import pandas as pd
from pathlib import Path

BASE_1995 = Path("./../data/datasets/1995")
APPROVED_DIR = BASE_1995 / "approved"

MAP_PATH = "./../data/Disease_burden_mapping/Mapping_diseases_disease_classes.csv"
GBD_PATH = "./../data/Disease_burden_mapping/Global_disease_burden_statistics_download.csv"

AGENCIES = ["EMA", "FDA", "Health Canada", "PMDA", "TGA", "Swissmedic"]

dfs = []
for agency in AGENCIES:
    fname = f"{agency.replace(' ', '')}.csv"   # HealthCanada.csv
    path = APPROVED_DIR / fname
    df = pd.read_csv(path)
    df["Agency"] = agency  # wichtig: sicherstellen, dass Agency korrekt gesetzt ist
    dfs.append(df)

APPROVALS_DF = pd.concat(dfs, ignore_index=True)

# Quick sanity
print(APPROVALS_DF["Agency"].value_counts())

# ==================================================
# Study years
# ==================================================

START_YEAR = 1995
END_YEAR = 2023

# ==================================================
# Canonical disease classes
# ==================================================

CANONICAL_CLASSES = [
    "Infectious and parasitic diseases",
    "Neoplasms",
    "Diseases of the blood and blood-forming organs",
    "Endocrine, nutritional, and metabolic diseases",
    "Mental and behavioural disorders",
    "Diseases of the nervous system",
    "Diseases of the eye and adnexa",
    "Diseases of the ear and mastoid process",
    "Diseases of the circulatory system",
    "Diseases of the respiratory system",
    "Diseases of the digestive system",
    "Diseases of the skin",
    "Diseases of the musculoskeletal system and connective tissue",
    "Diseases of the genitourinary system",
    "Pregnancy and childbirth",
    "Congenital malformations and chromosomal abnormalities",
    "Injury, poisoning and certain other consequences of external causes",
    "Other",
]

# ==================================================
# Helpers
# ==================================================

PLACEHOLDERS = {"not reported", "na", "n/a", "tbd", "none", ""}

def norm_str(s: pd.Series) -> pd.Series:
    return s.astype("string").str.strip()

def approved_mask_from_decision(decision_series: pd.Series) -> pd.Series:
    dec = decision_series.astype("string").str.lower().str.strip().fillna("")
    return dec.str.contains(r"\b(approved|authorised|authorized)\b", regex=True)

def normalise_disease_class(x) -> str:
    if pd.isna(x):
        return "Other"

    t = str(x).strip()
    if t.lower() in PLACEHOLDERS:
        return "Other"

    fixes = {
        "Diseases of the musculskeletal system and connective tissue":
            "Diseases of the musculoskeletal system and connective tissue",
        "Congenital malformations and chromosal abnormalities":
            "Congenital malformations and chromosomal abnormalities",
        "Injury, poisining and certain other consequences of external causes":
            "Injury, poisoning and certain other consequences of external causes",
    }

    t = fixes.get(t, t)

    if t in CANONICAL_CLASSES:
        return t

    lower_map = {c.lower(): c for c in CANONICAL_CLASSES}
    return lower_map.get(t.lower(), "Other")

# ==================================================
# Approvals aggregation (DEFINE FIRST)
# ==================================================

def approvals_by_disease_class(approvals_df: pd.DataFrame) -> pd.DataFrame:
    appr = approvals_df.copy()

    # KEIN decision filter mehr

    appr["Decision_year"] = pd.to_numeric(appr["Decision_year"], errors="coerce")
    appr = appr[appr["Decision_year"].between(START_YEAR, END_YEAR)]

    # Split & explode disease classes
    appr["Disease_class"] = appr["Disease_class(es)"].astype("string").str.split(";")
    appr = appr.explode("Disease_class")
    appr["Disease_class"] = appr["Disease_class"].astype("string").str.strip()
    appr = appr[appr["Disease_class"].notna() & (appr["Disease_class"] != "")]

    appr["Disease_class"] = appr["Disease_class"].map(normalise_disease_class)

    yearly = (
        appr.groupby(["Disease_class", "Decision_year"])
        .size()
        .reset_index(name="approvals_n")
    )

    summary = (
        yearly.groupby("Disease_class")
        .agg(
            mean_approvals_per_year=("approvals_n", "mean"),
            max_approvals_per_year=("approvals_n", "max"),
        )
        .reset_index()
    )

    peak_year = (
        yearly.sort_values(
            ["Disease_class", "approvals_n", "Decision_year"],
            ascending=[True, False, True]
        )
        .drop_duplicates("Disease_class")[["Disease_class", "Decision_year"]]
        .rename(columns={"Decision_year": "peak_year"})
    )

    summary = summary.merge(peak_year, on="Disease_class", how="left")

    total = len(appr)
    share = (
        appr["Disease_class"]
        .value_counts()
        .reindex(CANONICAL_CLASSES, fill_value=0)
        .reset_index(name="approvals_mentions_n")
        .rename(columns={"index": "Disease_class"})
    )

    share["approvals_mentions_pct"] = (
        share["approvals_mentions_n"] / total * 100
    ).round(2) if total else 0.0

    return summary.merge(
        share[["Disease_class", "approvals_mentions_pct"]],
        on="Disease_class",
        how="left"
    )

# ==================================================
# 1) Load mapping
# ==================================================

mapping_raw = pd.read_csv(MAP_PATH, header=None).rename(columns={0: "cause_name"})
mapping_long = mapping_raw.melt(id_vars=["cause_name"], value_name="Disease_class_raw")
mapping_long["cause_name"] = norm_str(mapping_long["cause_name"])
mapping_long["Disease_class"] = mapping_long["Disease_class_raw"].map(normalise_disease_class)
mapping_long = mapping_long.drop_duplicates(["cause_name", "Disease_class"])

# ==================================================
# 2) Load GBD
# ==================================================

gbd = pd.read_csv(GBD_PATH)
gbd["year"] = pd.to_numeric(gbd["year"], errors="coerce")
gbd["cause_name"] = norm_str(gbd["cause_name"])

# --- standardise measure names so pivot columns are always: Incidence / Prevalence / Deaths
gbd["measure_name"] = (
    gbd["measure_name"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({
        "incidence": "Incidence",
        "prevalence": "Prevalence",
        "deaths": "Deaths",
        "death": "Deaths",
    })
)

gbd_f = gbd[
    (gbd["measure_name"].isin(["Incidence", "Prevalence", "Deaths"])) &
    (gbd["metric_name"].str.lower() == "rate") &
    (gbd["year"].isin([START_YEAR, END_YEAR]))
]

# ==================================================
# 3) Aggregate disease burden
# ==================================================

gbd_mapped = gbd_f.merge(mapping_long, on="cause_name", how="left")
gbd_mapped["Disease_class"] = gbd_mapped["Disease_class"].fillna("Other")

burden = (
    gbd_mapped.groupby(["Disease_class", "measure_name", "year"])["val"]
    .sum()
    .reset_index()
)

burden_wide = (
    burden.pivot_table(
        index=["Disease_class", "measure_name"],
        columns="year",
        values="val"
    )
    .reset_index()
)

burden_wide["pct_change"] = (
    (burden_wide[END_YEAR] - burden_wide[START_YEAR]) /
    burden_wide[START_YEAR] * 100
)

burden_change_tbl = (
    burden_wide.pivot_table(
        index="Disease_class",
        columns="measure_name",
        values="pct_change"
    )
    .reindex(CANONICAL_CLASSES)
    .reset_index()
    .rename(columns={
        "Incidence": f"Incidence_pct_change_{START_YEAR}_{END_YEAR}",
        "Prevalence": f"Prevalence_pct_change_{START_YEAR}_{END_YEAR}",
        "Deaths": f"Deaths_pct_change_{START_YEAR}_{END_YEAR}",
    })
)

burden_end_tbl = (
    burden_wide.pivot_table(
        index="Disease_class",
        columns="measure_name",
        values=END_YEAR
    )
    .reindex(CANONICAL_CLASSES)
    .reset_index()
    .rename(columns={
        "Incidence": f"Incidence_{END_YEAR}",
        "Prevalence": f"Prevalence_{END_YEAR}",
        "Deaths": f"Deaths_{END_YEAR}",
    })
)

burden_class_summary = burden_change_tbl.merge(
    burden_end_tbl, on="Disease_class", how="left"
)

# ==================================================
# 4) PAR vs NON-PAR approvals
# ==================================================

appr_summary_par = approvals_by_disease_class(
    APPROVALS_DF[APPROVALS_DF["Agency"].isin(PAR_AGENCIES)]
)

appr_summary_nonpar = approvals_by_disease_class(
    APPROVALS_DF[APPROVALS_DF["Agency"].isin(NON_PAR_AGENCIES)]
)

# ==================================================
# 5) Combine (IDENTICAL STRUCTURE)
# ==================================================

combined_par = burden_class_summary.merge(
    appr_summary_par, on="Disease_class", how="left"
)

combined_nonpar = burden_class_summary.merge(
    appr_summary_nonpar, on="Disease_class", how="left"
)

for df in (combined_par, combined_nonpar):
    df["mean_approvals_per_year"] = df["mean_approvals_per_year"].fillna(0)
    df["max_approvals_per_year"] = df["max_approvals_per_year"].fillna(0)
    df["peak_year"] = df["peak_year"].fillna(pd.NA)

# ==================================================
# 6) OUTPUTS – IDENTICAL WIE ZUVOR
# ==================================================

import pandas as pd

candidates = [
    (name, obj)
    for name, obj in globals().items()
    if isinstance(obj, pd.DataFrame) and "Agency" in obj.columns
]

print("DataFrames with an 'Agency' column:")
for name, obj in candidates:
    vc = obj["Agency"].astype("string").str.strip().value_counts()
    print(f"\n{name}:")
    print(vc.head(10))

    if any(a in vc.index.tolist() for a in ["FDA", "Health Canada"]):
        print(">>> FOUND FDA / Health Canada here <<<")

# ---------- PAR ----------
print("\n=== PAR agencies: High-burden classes (top 10 by Deaths in 2021) ===")
display(
    combined_par.sort_values(f"Deaths_{END_YEAR}", ascending=False).head(10)
)

print("\n=== PAR agencies: Combined table (all disease classes) ===")
display(
    combined_par.set_index("Disease_class").reindex(CANONICAL_CLASSES).reset_index()
)

# ---------- NON-PAR ----------
print("\n=== NON-PAR agencies: High-burden classes (top 10 by Deaths in 2021) ===")
display(
    combined_nonpar.sort_values(f"Deaths_{END_YEAR}", ascending=False).head(10)
)

print("\n=== NON-PAR agencies: Combined table (all disease classes) ===")
display(
    combined_nonpar.set_index("Disease_class").reindex(CANONICAL_CLASSES).reset_index()
)

Agency
FDA              18558
Health Canada    10286
EMA               1491
TGA                988
PMDA               408
Swissmedic         233
Name: count, dtype: int64
DataFrames with an 'Agency' column:

df_ema:
Agency
EMA    1491
Name: count, dtype: Int64

df_swissmedic:
Agency
Swissmedic    233
Name: count, dtype: Int64

df_japan:
Agency
PMDA    408
Name: count, dtype: Int64

df_australia:
Agency
TGA    988
Name: count, dtype: Int64

record_summary:
Agency
EMA           1
Swissmedic    1
Japan         1
Australia     1
Name: count, dtype: Int64

approved_record_summary:
Agency
EMA           1
Swissmedic    1
Japan         1
Australia     1
Name: count, dtype: Int64

ma_summary:
Agency
EMA           1
Swissmedic    1
Japan         1
Australia     1
Name: count, dtype: Int64

per_agency_df:
Agency
EMA              1
FDA              1
Health Canada    1
PMDA             1
TGA              1
Swissmedic       1
Name: count, dtype: Int64
>>> FOUND FDA / Health Canada here <<<

subset:

,Disease_class,Deaths_pct_change_1995_2023,Incidence_pct_change_1995_2023,Prevalence_pct_change_1995_2023,Deaths_2023,Incidence_2023,Prevalence_2023,mean_approvals_per_year,max_approvals_per_year,peak_year,approvals_mentions_pct
17,Other,-20.970134,17.284030,-14.519818,405.588541,327685.940517,198433.071894,2.090909,5,2023,0.56
8,Diseases of the circulatory system,8.594791,23.534173,33.367330,148.211286,475.258468,7510.683834,11.000000,34,2009,5.91
9,Diseases of the respiratory system,16.728192,-1.130032,15.654907,51.506827,1035.179498,7226.020893,19.055556,42,2021,8.37
10,Diseases of the digestive system,-48.824842,-22.099353,7.329844,42.531892,103180.022379,82623.958964,9.476190,23,2023,4.86
4,Mental and behavioural disorders,72.635775,12.463844,24.288475,31.644716,6001.910102,18346.455126,4.944444,15,2015,2.17
13,Diseases of the genitourinary system,57.638834,6.403872,12.390739,20.488736,11527.155974,34009.583283,8.761905,22,2022,4.49
1,Neoplasms,28.464971,-2.990852,4.783738,19.502596,1202.839907,2468.560621,35.739130,79,2020,20.06
0,Infectious and parasitic diseases,-45.374753,-30.771690,-16.074005,19.106114,4170.171739,4543.407651,21.000000,59,2022,11.79
5,Diseases of the nervous system,47.952691,16.746566,25.418477,10.547277,57.422679,501.707500,12.043478,33,2021,6.76
14,Pregnancy and childbirth,-62.803917,-25.639925,13.361874,8.956966,1251.323347,3740.035134,1.416667,4,2021,0.41



=== PAR agencies: Combined table (all disease classes) ===


,Disease_class,Deaths_pct_change_1995_2023,Incidence_pct_change_1995_2023,Prevalence_pct_change_1995_2023,Deaths_2023,Incidence_2023,Prevalence_2023,mean_approvals_per_year,max_approvals_per_year,peak_year,approvals_mentions_pct
0,Infectious and parasitic diseases,-45.374753,-30.771690,-16.074005,19.106114,4170.171739,4543.407651,21.000000,59,2022,11.79
1,Neoplasms,28.464971,-2.990852,4.783738,19.502596,1202.839907,2468.560621,35.739130,79,2020,20.06
2,Diseases of the blood and blood-forming organs,-18.461694,NaN,-15.178334,0.643067,0.000000,1615.732337,17.538462,52,2023,11.13
3,"Endocrine, nutritional, and metabolic diseases",NaN,NaN,NaN,NaN,NaN,NaN,16.208333,37,2022,9.49
4,Mental and behavioural disorders,72.635775,12.463844,24.288475,31.644716,6001.910102,18346.455126,4.944444,15,2015,2.17
5,Diseases of the nervous system,47.952691,16.746566,25.418477,10.547277,57.422679,501.707500,12.043478,33,2021,6.76
6,Diseases of the eye and adnexa,NaN,NaN,37.218290,NaN,NaN,9661.408151,4.850000,14,2022,2.37
7,Diseases of the ear and mastoid process,NaN,NaN,40.139109,NaN,NaN,21092.218962,1.000000,1,2013,0.02
8,Diseases of the circulatory system,8.594791,23.534173,33.367330,148.211286,475.258468,7510.683834,11.000000,34,2009,5.91
9,Diseases of the respiratory system,16.728192,-1.130032,15.654907,51.506827,1035.179498,7226.020893,19.055556,42,2021,8.37



=== NON-PAR agencies: High-burden classes (top 10 by Deaths in 2021) ===


,Disease_class,Deaths_pct_change_1995_2023,Incidence_pct_change_1995_2023,Prevalence_pct_change_1995_2023,Deaths_2023,Incidence_2023,Prevalence_2023,mean_approvals_per_year,max_approvals_per_year,peak_year,approvals_mentions_pct
17,Other,-20.970134,17.284030,-14.519818,405.588541,327685.940517,198433.071894,4.230769,32,2013,0.58
8,Diseases of the circulatory system,8.594791,23.534173,33.367330,148.211286,475.258468,7510.683834,79.413793,209,2012,12.21
9,Diseases of the respiratory system,16.728192,-1.130032,15.654907,51.506827,1035.179498,7226.020893,43.931034,79,2021,6.76
10,Diseases of the digestive system,-48.824842,-22.099353,7.329844,42.531892,103180.022379,82623.958964,44.172414,85,2022,6.79
4,Mental and behavioural disorders,72.635775,12.463844,24.288475,31.644716,6001.910102,18346.455126,51.000000,103,2018,7.84
13,Diseases of the genitourinary system,57.638834,6.403872,12.390739,20.488736,11527.155974,34009.583283,52.724138,97,2019,8.11
1,Neoplasms,28.464971,-2.990852,4.783738,19.502596,1202.839907,2468.560621,44.379310,149,2021,6.82
0,Infectious and parasitic diseases,-45.374753,-30.771690,-16.074005,19.106114,4170.171739,4543.407651,51.344828,99,2018,7.90
5,Diseases of the nervous system,47.952691,16.746566,25.418477,10.547277,57.422679,501.707500,65.724138,165,2013,10.11
14,Pregnancy and childbirth,-62.803917,-25.639925,13.361874,8.956966,1251.323347,3740.035134,11.586207,25,2020,1.78



=== NON-PAR agencies: Combined table (all disease classes) ===


,Disease_class,Deaths_pct_change_1995_2023,Incidence_pct_change_1995_2023,Prevalence_pct_change_1995_2023,Deaths_2023,Incidence_2023,Prevalence_2023,mean_approvals_per_year,max_approvals_per_year,peak_year,approvals_mentions_pct
0,Infectious and parasitic diseases,-45.374753,-30.771690,-16.074005,19.106114,4170.171739,4543.407651,51.344828,99,2018,7.90
1,Neoplasms,28.464971,-2.990852,4.783738,19.502596,1202.839907,2468.560621,44.379310,149,2021,6.82
2,Diseases of the blood and blood-forming organs,-18.461694,NaN,-15.178334,0.643067,0.000000,1615.732337,28.827586,85,2021,4.43
3,"Endocrine, nutritional, and metabolic diseases",NaN,NaN,NaN,NaN,NaN,NaN,56.413793,145,2023,8.67
4,Mental and behavioural disorders,72.635775,12.463844,24.288475,31.644716,6001.910102,18346.455126,51.000000,103,2018,7.84
5,Diseases of the nervous system,47.952691,16.746566,25.418477,10.547277,57.422679,501.707500,65.724138,165,2013,10.11
6,Diseases of the eye and adnexa,NaN,NaN,37.218290,NaN,NaN,9661.408151,15.310345,37,2023,2.35
7,Diseases of the ear and mastoid process,NaN,NaN,40.139109,NaN,NaN,21092.218962,5.428571,13,2020,0.81
8,Diseases of the circulatory system,8.594791,23.534173,33.367330,148.211286,475.258468,7510.683834,79.413793,209,2012,12.21
9,Diseases of the respiratory system,16.728192,-1.130032,15.654907,51.506827,1035.179498,7226.020893,43.931034,79,2021,6.76


In [30]:
os.getcwd()

'/Users/jacquelinedort/Documents/DrugFork/src'

In [31]:
sorted(gbd["year"].unique())[-10:]

[np.int64(2014),
 np.int64(2015),
 np.int64(2016),
 np.int64(2017),
 np.int64(2018),
 np.int64(2019),
 np.int64(2020),
 np.int64(2021),
 np.int64(2022),
 np.int64(2023)]

## Overlapping Drug Names throughout agencies (case-insensitive match)

In [32]:
import pandas as pd

# =========================
# 1. Daten laden & bereinigen
# =========================

df = pd.read_csv("./../data/datasets/1995/all_decisions/Overall.csv")

df = df[
    ["Non_proprietary_name", "Drug_name", "Dataset"]
]

df["Non_proprietary_name"] = (
    df["Non_proprietary_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["Drug_name"] = (
    df["Drug_name"]
    .astype(str)
    .str.strip()
)

df = df.drop_duplicates()

# =========================
# 2. Wide table: Drug names pro Agency
# =========================

grouped = (
    df
    .groupby(["Non_proprietary_name", "Dataset"])["Drug_name"]
    .apply(lambda x: sorted(set(x)))
    .reset_index()
)

pivot = grouped.pivot(
    index="Non_proprietary_name",
    columns="Dataset",
    values="Drug_name"
)

pivot["n_unique_drug_names"] = pivot.apply(
    lambda row: len(set(
        name
        for cell in row.dropna()
        for name in cell
    )),
    axis=1
)

pivot.to_csv(
    "./../output/overlap/drug_names_per_Substance_per_agency.csv"
)

# =========================
# 3. Anzahl Agencies pro INN
# =========================

agency_count = (
    df
    .groupby("Non_proprietary_name")["Dataset"]
    .nunique()
    .rename("n_agencies")
    .reset_index()
)

# =========================
# 4. Drug name → Anzahl Agencies
# =========================

name_agency_map = (
    df
    .groupby(["Non_proprietary_name", "Drug_name"])["Dataset"]
    .nunique()
    .reset_index(name="n_agencies_per_name")
)

# =========================
# 5. Prüfen: global konsistenter Name?
# =========================

name_agency_map = name_agency_map.merge(
    agency_count,
    on="Non_proprietary_name",
    how="left"
)

name_agency_map["is_globally_consistent"] = (
    name_agency_map["n_agencies_per_name"]
    == name_agency_map["n_agencies"]
)

# =========================
# 6. Summary pro INN
# =========================

summary = (
    name_agency_map
    .groupby("Non_proprietary_name")
    .agg(
        n_agencies=("n_agencies", "first"),
        drug_names=("Drug_name", lambda x: sorted(set(x))),
        has_globally_consistent_name=("is_globally_consistent", "any")
    )
    .reset_index()
)

summary.to_csv(
    "./../output/overlap/drug_name_consistency_summary.csv",
    index=False
)

# =========================
# 7. Zusatz-Output: detailliertes Mapping
# =========================

name_agency_map.to_csv(
    "./../output/overlap/drug_name_origin_by_agency_count.csv",
    index=False
)

# Absolute Zahlen
n_total = summary.shape[0]
n_consistent = summary["has_globally_consistent_name"].sum()

# Prozent
percent_consistent = (n_consistent / n_total) * 100

print(f"Total number of substances (INNs): {n_total}")
print(f"Substances with ≥1 globally consistent drug name: {n_consistent}")
print(f"Proportion with globally consistent name: {percent_consistent:.1f}%")




Total number of substances (INNs): 4655
Substances with ≥1 globally consistent drug name: 4075
Proportion with globally consistent name: 87.5%


Anzahl Substanzen mit mehr als einem Produktnamen

In [33]:
(pivot["n_unique_drug_names"] > 1).mean()


np.float64(0.3727975934679845)